In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import read, UTCDateTime
from obspy.core import Trace, Stream
from PyEMD import EEMD

In [ ]:
# --- Config ---
input_dir = "C:/work/all_paper/papers/p3/filtered_waveforms_2/FINAL_RUN/DATA_FINAL/z/"
output_dir = "C:/work/all_paper/papers/p3/filtered_waveforms_2/eemd2/"
os.makedirs(output_dir, exist_ok=True)

In [3]:
# --- SNR Calculation ---
def compute_snr(signal, noise_window=200):
    """Compute SNR based on first `noise_window` samples as noise."""
    signal_power = np.mean(signal ** 2)
    noise_power = np.mean(signal[:noise_window] ** 2)
    return signal_power / noise_power if noise_power > 0 else 1e-6


In [ ]:
# --- Process Single Trace ---
def process_trace(trace, out_path):
    data = trace.data.astype(np.float64)

    eemd = EEMD()
    imfs = eemd.eemd(data)

    original_snr = compute_snr(data)
    imf_snrs = [compute_snr(imf) for imf in imfs]

    nrows = len(imfs) + 1
    fig, axs = plt.subplots(
        nrows, 1,
        figsize=(12, 2 * nrows),
        sharex=True
    )

    axs = np.atleast_1d(axs)

    # --- Original signal ---
    axs[0].plot(data, color="black", linewidth=0.8, label="Original Signal")
    axs[0].text(
        0.01, 0.88,
        f"SNR = {original_snr:.2f}",
        transform=axs[0].transAxes,
        fontsize=10,
        verticalalignment="top"
    )
    axs[0].legend(loc="upper right", fontsize=9)
    axs[0].set_yticks([])          
    axs[0].set_ylabel("")          
    axs[0].grid(True, alpha=0.3)

    # --- IMFs ---
    for i, (imf, snr) in enumerate(zip(imfs, imf_snrs), start=1):
        axs[i].plot(imf, linewidth=0.8, label=f"IMF {i}")
        axs[i].text(
            0.01, 0.88,
            f"SNR = {snr:.2f}",
            transform=axs[i].transAxes,
            fontsize=10,
            verticalalignment="top"
        )
        axs[i].legend(loc="upper right", fontsize=9)
        axs[i].set_yticks([])          
        axs[i].set_ylabel("")          
        axs[i].grid(True, alpha=0.3)

    axs[-1].set_xlabel("Sample index")
    
    fig.suptitle(f"EEMD Decomposition: {trace.id}", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.97])

    fig.savefig(out_path, dpi=450, bbox_inches="tight")
    plt.close(fig)

In [ ]:
# --- Main Loop Over Folder ---
for file_name in os.listdir(input_dir):
    if not file_name.endswith(".mseed"):
        continue

    file_path = os.path.join(input_dir, file_name)
    try:
        stream = read(file_path)
        for trace in stream:
            trace_id = trace.id.replace(".", "_")
            out_file = os.path.join(output_dir, f"{trace_id}.png")
            print(f"🔍 Processing {trace.id}")
            process_trace(trace, out_file)
    except Exception as e:
        print(f"Failed to process {file_name}: {e}")

print("All done. EEMD plots saved in output folder.")

IMD Energy Calculation

In [ ]:

MSEED_FILE = r"C:/work/all_paper/papers/p3/filtered_waveforms/data/2025058_058000_NIZM_Z2345.mseed"

EVENT_NAME = "Mana Avalanche"
START_TIME = "2025-02-27T23:45:00"
END_TIME   = "2025-02-27T23:55:00"

COMPONENT = "Z"

st = read(MSEED_FILE)
st = st.select(component=COMPONENT)

tr = st[0].copy()
tr.trim(
    starttime=UTCDateTime(START_TIME),
    endtime=UTCDateTime(END_TIME),
    pad=False
)

data = tr.data.astype(np.float64)

eemd = EEMD()
eemd.noise_seed(42)
imfs = eemd.eemd(data)

# IMF ENERGY
imf_energy = np.sum(imfs ** 2, axis=1)
imf_energy_percent = 100 * imf_energy / np.sum(imf_energy)

max_imf_index = np.argmax(imf_energy_percent)
max_imf = max_imf_index + 1
max_energy_percent = imf_energy_percent[max_imf_index]

df = pd.DataFrame({
    "IMF": [f"IMF-{i+1}" for i in range(len(imfs))],
    "Energy": imf_energy,
    "Energy (%)": np.round(imf_energy_percent, 2)
})

print(df)
print(f"\nIMF carrying maximum energy: IMF-{max_imf} ({max_energy_percent:.2f}%)")

df.to_csv(f"{EVENT_NAME.replace(' ', '_')}_IMF_energy.csv", index=False)

       IMF        Energy  Energy (%)
0    IMF-1  2.572136e+18        0.55
1    IMF-2  9.630727e+17        0.21
2    IMF-3  6.256932e+17        0.13
3    IMF-4  8.941352e+18        1.91
4    IMF-5  2.091233e+20       44.65
5    IMF-6  2.031866e+20       43.38
6    IMF-7  3.979424e+19        8.50
7    IMF-8  2.125171e+18        0.45
8    IMF-9  4.864686e+17        0.10
9   IMF-10  3.358304e+17        0.07
10  IMF-11  9.616057e+16        0.02
11  IMF-12  3.488678e+16        0.01
12  IMF-13  1.829091e+16        0.00
13  IMF-14  2.346313e+16        0.01
14  IMF-15  2.875298e+15        0.00
15  IMF-16  8.813371e+13        0.00
16  IMF-17  9.323767e+15        0.00

IMF carrying maximum energy: IMF-5 (44.65%)


In [ ]:

MSEED_FILE = r"C:/work/all_paper/papers/p3/filtered_waveforms_2/FINAL_RUN/DATA_FINAL/z/2025089_089000_SANK_Z1129.mseed"

EVENT_NAME = "Mnikaran landslide"
START_TIME = "2025-03-30T11:30:01"
END_TIME   = "2025-03-30T11:32:57.11"

COMPONENT = "Z"

st = read(MSEED_FILE)
st = st.select(component=COMPONENT)

tr = st[0].copy()
tr.trim(
    starttime=UTCDateTime(START_TIME),
    endtime=UTCDateTime(END_TIME),
    pad=False
)

data = tr.data.astype(np.float64)

eemd = EEMD()
eemd.noise_seed(42)

imfs = eemd.eemd(data)

# IMF ENERGY
imf_energy = np.sum(imfs ** 2, axis=1)
imf_energy_percent = 100 * imf_energy / np.sum(imf_energy)

max_imf_index = np.argmax(imf_energy_percent)
max_imf = max_imf_index + 1
max_energy_percent = imf_energy_percent[max_imf_index]

df = pd.DataFrame({
    "IMF": [f"IMF-{i+1}" for i in range(len(imfs))],
    "Energy": imf_energy,
    "Energy (%)": np.round(imf_energy_percent, 2)
})

print(df)
print(f"\nIMF carrying maximum energy: IMF-{max_imf} ({max_energy_percent:.2f}%)")

df.to_csv(f"{EVENT_NAME.replace(' ', '_')}_IMF_energy1.csv", index=False)

       IMF        Energy  Energy (%)
0    IMF-1  1.036969e+20       91.24
1    IMF-2  9.267263e+18        8.15
2    IMF-3  3.289310e+17        0.29
3    IMF-4  1.408029e+17        0.12
4    IMF-5  7.128847e+16        0.06
5    IMF-6  4.301448e+16        0.04
6    IMF-7  2.319637e+16        0.02
7    IMF-8  1.210733e+16        0.01
8    IMF-9  5.743206e+15        0.01
9   IMF-10  2.453016e+15        0.00
10  IMF-11  1.178023e+15        0.00
11  IMF-12  1.736939e+14        0.00
12  IMF-13  9.637616e+14        0.00
13  IMF-14  1.510232e+15        0.00
14  IMF-15  5.397778e+16        0.05

IMF carrying maximum energy: IMF-1 (91.24%)


In [ ]:

MSEED_FILE = r"C:/work/all_paper/papers/p3/earthquake/ayar/z/2022189_189000_AYAR_Z0101.mseed"

EVENT_NAME = "Earthquake"
START_TIME = "2022-07-08T01:01:46.16"
END_TIME   = "2022-07-08T01:04:00"

COMPONENT = "Z"

st = read(MSEED_FILE)
st = st.select(component=COMPONENT)

tr = st[0].copy()
tr.trim(
    starttime=UTCDateTime(START_TIME),
    endtime=UTCDateTime(END_TIME),
    pad=False
)

data = tr.data.astype(np.float64)

eemd = EEMD()
eemd.noise_seed(42)
imfs = eemd.eemd(data)

# IMF ENERGY
imf_energy = np.sum(imfs ** 2, axis=1)
imf_energy_percent = 100 * imf_energy / np.sum(imf_energy)

max_imf_index = np.argmax(imf_energy_percent)
max_imf = max_imf_index + 1
max_energy_percent = imf_energy_percent[max_imf_index]

df = pd.DataFrame({
    "IMF": [f"IMF-{i+1}" for i in range(len(imfs))],
    "Energy": imf_energy,
    "Energy (%)": np.round(imf_energy_percent, 2)
})

print(df)
print(f"\nIMF carrying maximum energy: IMF-{max_imf} ({max_energy_percent:.2f}%)")

df.to_csv(f"{EVENT_NAME.replace(' ', '_')}_IMF_energy.csv", index=False)

       IMF        Energy  Energy (%)
0    IMF-1  6.298360e+17        1.02
1    IMF-2  5.140152e+17        0.83
2    IMF-3  2.979537e+19       48.18
3    IMF-4  2.290424e+19       37.03
4    IMF-5  6.600944e+18       10.67
5    IMF-6  1.324288e+18        2.14
6    IMF-7  3.687692e+16        0.06
7    IMF-8  1.649878e+16        0.03
8    IMF-9  1.050138e+16        0.02
9   IMF-10  5.869035e+15        0.01
10  IMF-11  3.137104e+15        0.01
11  IMF-12  1.314077e+15        0.00
12  IMF-13  3.305772e+14        0.00
13  IMF-14  4.305401e+15        0.01

IMF carrying maximum energy: IMF-3 (48.18%)
